# PIUBAMAS DNU — Análisis de RACISM (marzo 2021)

Natalia Debandi

Exploración del dataset `piubamas_dnu_marzo2021.csv` con foco en la etiqueta **RACISM**.

| Sección | Contenido |
|---|---|
| 3 | Serie temporal — % RACISM por día |
| 4 | % RACISM por medio |
| 5 | Noticias con más RACISM + comentarios aleatorios |
| 6 | Nube de palabras |
| 7 | Exportar HTML |

## 1. Imports y configuración

In [ ]:
import re
import html as htmllib
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

FILE = 'data/piubamas_dnu_marzo2021.csv'

LABEL_COLS = ['WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS',
              'DISABLED', 'APPEARANCE', 'CRIMINAL', 'CALLS']

LABEL_COLORS = {
    'RACISM':    '#c0392b',
    'CRIMINAL':  '#7c3aed',
    'CALLS':     '#ea580c',
    'POLITICS':  '#2563eb',
    'WOMEN':     '#db2777',
    'LGBTI':     '#0891b2',
    'CLASS':     '#65a30d',
    'DISABLED':  '#d97706',
    'APPEARANCE':'#6b7280',
    'HATEFUL':   '#dc2626',
}

MEDIO_COLORS = {
    'infobae':        '#e11d48',
    'clarincom':      '#1d4ed8',
    'LANACION':       '#b45309',
    'pagina12':       '#15803d',
    'cronica':        '#7c3aed',
    'perfilcom':      '#0e7490',
    'laderechadiario':'#9a3412',
    'izquierdadiario':'#be185d',
}

def badge(label):
    color = LABEL_COLORS.get(label, '#374151')
    return (f'<span style="background:{color};color:white;border-radius:4px;'
            f'padding:2px 7px;font-size:11px;margin:2px;display:inline-block">{label}</span>')

## 2. Carga de datos

In [ ]:
df = pd.read_csv(FILE, low_memory=False)
df['fecha']     = pd.to_datetime(df['date_tweet']).dt.normalize()
df['fecha_dia'] = df['fecha'].dt.date
df[LABEL_COLS]  = df[LABEL_COLS].astype(int)

HATEFUL_LABELS = [l for l in LABEL_COLS if l != 'CALLS']
df['HATEFUL']  = df[HATEFUL_LABELS].any(axis=1).astype(int)

n_total  = len(df)
n_racism = (df['RACISM'] == 1).sum()

print(f'Dataset:           {n_total:,} comentarios')
print(f'Con RACISM=1:      {n_racism:,}  ({n_racism/n_total*100:.2f}%)')
print(f'Noticias únicas:   {df["tweet_id_noticia"].nunique():,}')
print(f'Medios:            {sorted(df["medio"].unique())}')
print(f'Período:           {df["fecha_dia"].min()} → {df["fecha_dia"].max()}')

## 3. Serie temporal — % RACISM por día

In [ ]:
por_dia = (
    df.groupby('fecha_dia')
    .agg(total=('id', 'count'), n_racism=('RACISM', 'sum'))
    .reset_index()
)
por_dia['% RACISM']  = (por_dia['n_racism'] / por_dia['total'] * 100).round(2)
por_dia['fecha_dia'] = pd.to_datetime(por_dia['fecha_dia'])

display(
    por_dia.set_index('fecha_dia')
    .style
    .format({'total': '{:,}', 'n_racism': '{:,}', '% RACISM': '{:.2f}%'})
    .background_gradient(subset=['% RACISM'], cmap='RdPu')
    .background_gradient(subset=['total'],    cmap='Blues')
    .set_caption('Comentarios y tasa de RACISM por día')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '13px'), ('font-weight', 'bold'), ('text-align', 'left')]}])
)

fig, ax1 = plt.subplots(figsize=(11, 4))

ax1.bar(por_dia['fecha_dia'], por_dia['total'],
        color='#bfdbfe', width=0.6, label='Total comentarios', zorder=1)
ax1.set_ylabel('Comentarios totales', color='#1e40af', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#1e40af')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))

ax2 = ax1.twinx()
ax2.plot(por_dia['fecha_dia'], por_dia['% RACISM'],
         color='#c0392b', marker='o', linewidth=2.2, markersize=6,
         label='% RACISM', zorder=2)
ax2.set_ylabel('% RACISM', color='#c0392b', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#c0392b')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

for _, row in por_dia.iterrows():
    ax2.annotate(f"{row['% RACISM']:.1f}%",
                 xy=(row['fecha_dia'], row['% RACISM']),
                 xytext=(0, 7), textcoords='offset points',
                 ha='center', fontsize=8, color='#c0392b')

ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%d %b'))
ax1.set_title('Tasa de RACISM diaria — comentarios a noticias DNU (marzo 2021)', fontsize=13)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/racism_serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. % RACISM por medio

In [ ]:
por_medio = (
    df.groupby('medio')
    .agg(
        total      = ('id',               'count'),
        n_racism   = ('RACISM',           'sum'),
        n_noticias = ('tweet_id_noticia', 'nunique'),
    )
    .reset_index()
)
por_medio['% RACISM'] = (por_medio['n_racism'] / por_medio['total'] * 100).round(2)
por_medio = por_medio.sort_values('% RACISM', ascending=False)

display(
    por_medio.set_index('medio')
    .style
    .format({'total': '{:,}', 'n_racism': '{:,}', 'n_noticias': '{:,}', '% RACISM': '{:.2f}%'})
    .background_gradient(subset=['% RACISM'], cmap='RdPu')
    .background_gradient(subset=['total'],    cmap='Blues')
    .set_caption('Comentarios y tasa de RACISM por medio')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '13px'), ('font-weight', 'bold'), ('text-align', 'left')]}])
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
colors = [MEDIO_COLORS.get(m, '#6b7280') for m in por_medio['medio']]
bars = ax.barh(por_medio['medio'], por_medio['% RACISM'], color=colors, height=0.6, alpha=0.85)
for bar, val in zip(bars, por_medio['% RACISM']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2, f'{val:.2f}%', va='center', fontsize=9)
ax.set_xlabel('% RACISM')
ax.set_title('Tasa de RACISM por medio', fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis()

por_medio_vol = por_medio.sort_values('total', ascending=False)
ax2 = axes[1]
colors2 = [MEDIO_COLORS.get(m, '#6b7280') for m in por_medio_vol['medio']]
bars2 = ax2.barh(por_medio_vol['medio'], por_medio_vol['total'], color=colors2, height=0.6, alpha=0.85)
for bar, val in zip(bars2, por_medio_vol['total']):
    ax2.text(val + 20, bar.get_y() + bar.get_height() / 2, f'{val:,}', va='center', fontsize=9)
ax2.set_xlabel('Comentarios')
ax2.set_title('Volumen de comentarios por medio', fontsize=12)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax2.invert_yaxis()

fig.suptitle('RACISM por medio — tasa y volumen', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/racism_por_medio.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Noticias con más RACISM — comentarios aleatorios

Se muestran las **3 noticias** con mayor cantidad de comentarios RACISM.  
Para cada una: encabezado de la noticia + **10 comentarios aleatorios** (con o sin etiqueta).  
Ajustá `SEED` para explorar distintos comentarios; los comentarios con **borde rojo** tienen `RACISM=1`.

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────────
SEED          = 42    # ← cambiá este valor para ver distintos comentarios
N_NOTICIAS    = 3
N_COMENTARIOS = 10

# ── Top noticias por n_racism ──────────────────────────────────────────────────
top_noticias = (
    df.groupby('tweet_id_noticia')
    .agg(
        total     = ('id',               'count'),
        n_racism  = ('RACISM',           'sum'),
        n_hateful = ('HATEFUL',          'sum'),
        medio     = ('medio',            'first'),
        resumen   = ('resumen',          'first'),
        title     = ('title',            'first'),
        fecha     = ('fecha_dia',        'min'),
    )
    .sort_values('n_racism', ascending=False)
    .head(N_NOTICIAS)
)

# ── Tarjeta de comentario ─────────────────────────────────────────────────────
def comentario_card(row, idx):
    activas = [l for l in LABEL_COLS if row[l] == 1]
    badges  = ''.join(badge(l) for l in activas) or '<i style="color:#9ca3af">sin etiqueta</i>'
    texto   = htmllib.escape(str(row['text'])[:280])
    fecha   = str(row['fecha_dia'])
    border  = '2px solid #c0392b' if row['RACISM'] == 1 else '1px solid #e5e7eb'
    bg      = '#fff5f5' if row['RACISM'] == 1 else '#ffffff'
    return (
        f'<div style="background:{bg};border:{border};border-radius:8px;'
        f'padding:10px 14px;margin:5px 0">'
        f'<div style="font-size:11px;color:#6b7280;margin-bottom:4px">'
        f'💬 #{idx+1} &nbsp; 📅 {fecha}</div>'
        f'<div style="font-size:13px;color:#111;line-height:1.5;margin-bottom:6px">{texto}</div>'
        f'<div>{badges}</div>'
        f'</div>'
    )

# ── Renderizar noticias ────────────────────────────────────────────────────────
html_parts = []
for rank, (tweet_id, noticia) in enumerate(top_noticias.iterrows()):
    titulo  = htmllib.escape(str(noticia['title'])) if pd.notna(noticia['title']) else ''
    resumen = htmllib.escape(str(noticia['resumen'])[:220])
    medio   = noticia['medio']
    color   = MEDIO_COLORS.get(medio, '#374151')
    fecha   = str(noticia['fecha'])

    comentarios = df[df['tweet_id_noticia'] == tweet_id].copy()
    muestra = comentarios.sample(min(N_COMENTARIOS, len(comentarios)), random_state=SEED)

    cards_html = ''.join(
        comentario_card(row, i) for i, (_, row) in enumerate(muestra.iterrows())
    )

    titulo_bloque = (
        f'<div style="font-size:15px;font-weight:bold;color:#111;margin-bottom:6px;'
        f'line-height:1.3">{titulo}</div>' if titulo else ''
    )

    html_parts.append(
        f'<div style="font-family:Arial;border:2px solid #e2e8f0;border-radius:12px;'
        f'padding:18px;margin:22px 0">'
        f'<div style="display:flex;align-items:center;gap:10px;margin-bottom:12px">'
        f'<span style="background:{color};color:#fff;border-radius:6px;padding:3px 10px;'
        f'font-size:12px;font-weight:bold">{medio}</span>'
        f'<span style="font-size:12px;color:#6b7280">📅 {fecha}</span>'
        f'<span style="margin-left:auto;font-size:12px;color:#6b7280">'
        f'🗨️ {int(noticia["total"]):,} comentarios &nbsp;|&nbsp; '
        f'🔴 {int(noticia["n_racism"])} con RACISM</span>'
        f'</div>'
        f'{titulo_bloque}'
        f'<div style="font-size:12px;color:#374151;background:#f8fafc;border-radius:6px;'
        f'padding:8px 12px;margin-bottom:14px;line-height:1.5">{resumen}…</div>'
        f'<div style="font-size:12px;color:#94a3b8;margin-bottom:8px">'
        f'Mostrando {len(muestra)} comentarios aleatorios — SEED={SEED} '
        f'<span style="font-size:11px">(borde rojo = RACISM=1)</span></div>'
        + cards_html
        + '</div>'
    )

display(HTML('\n'.join(html_parts)))

## 6. Nube de palabras

- **Izquierda:** todo el dataset (vocabulario general de los comentarios)
- **Derecha:** solo comentarios con `RACISM=1`

In [ ]:
try:
    from wordcloud import WordCloud
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', 'wordcloud'], check=True)
    from wordcloud import WordCloud

STOPWORDS_WC = {
    'a','al','ante','bajo','con','contra','de','del','desde','durante','e','el','ella',
    'ellas','ellos','en','entre','esa','esas','ese','eso','esos','esta','estas','este',
    'esto','estos','hacia','hasta','la','las','le','les','lo','los','me','mi','mis',
    'ni','no','nos','o','para','pero','por','que','qué','se','si','sí','sin','sobre',
    'su','sus','te','tu','tú','un','una','uno','unas','unos','ya','yo','ser','estar',
    'tener','hacer','poder','querer','saber','ir','fue','fueron','era','eran','es','son',
    'hay','hubo','han','ha','he','tiene','tienen','tengo','van','va','ver','hizo','hace',
    'hacen','iba','iban','dijo','dice','deben','debe','pueden','puede','quiere','quieren',
    'así','bien','igual','mismo','tampoco','aunque','además','porque','cuando','donde',
    'aquí','acá','allá','allí','hoy','ahora','antes','después','siempre','nunca','cada',
    'más','muy','nada','sino','mucho','muchos','poco','tanto','tal','vez','otro','otros',
    'otra','otras','algunas','algunos','todo','todos','toda','también','tan','menos',
    'casi','solo','sólo','quien','como','quién','che','vos','dale','re','onda','tipo',
    'loco','loca','bueno','bue','claro','capaz','obvio','posta','jaja','jajaja','jajajaja',
    'jeje','jajaj','jaj','ojo','porfa','osea','pff','uff','ugh','mmm','aca','alla','ahi',
    'encima','rt','via','amp','co','clarincom','lanacion','infobae','pagina','cronica',
    'perfilcom',
}

def limpiar(texto):
    texto = str(texto).lower()
    texto = re.sub(r'http\S+|www\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'#\w+', '', texto)
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\b\w{1,2}\b', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

corpus_all = ' '.join(df['text'].apply(limpiar))
words_all  = [w for w in corpus_all.split() if w not in STOPWORDS_WC and len(w) >= 3]

corpus_rac = ' '.join(df[df['RACISM'] == 1]['text'].apply(limpiar))
words_rac  = [w for w in corpus_rac.split() if w not in STOPWORDS_WC and len(w) >= 3]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, words, title, cmap in [
    (axes[0], words_all, f'Todo el dataset ({len(df):,} comentarios)', 'Blues'),
    (axes[1], words_rac, f'Solo RACISM=1 ({(df["RACISM"]==1).sum():,} comentarios)', 'RdPu'),
]:
    wc = WordCloud(
        width=900, height=480,
        background_color='white',
        colormap=cmap,
        max_words=120,
        min_word_length=3,
        collocations=False,
        prefer_horizontal=0.85,
    ).generate(' '.join(words))
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.axis('off')

fig.suptitle('Nube de palabras — comentarios a noticias DNU marzo 2021', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/racism_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

top20 = Counter(words_all).most_common(20)
words_t, counts_t = zip(*top20)

fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(words_t) - 1, -1, -1)
ax.barh(list(y), counts_t, color='#3b82f6', height=0.7)
ax.set_yticks(list(y))
ax.set_yticklabels(words_t, fontsize=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title('Top 20 palabras más frecuentes — dataset completo', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/racism_top_palabras.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exportar HTML

In [ ]:
import nbformat
from nbconvert import HTMLExporter
from datetime import date

nb_path  = '4_PIUBA_RACISM.ipynb'
out_path = f'outputs/4_PIUBA_RACISM_{date.today()}.html'

with open(nb_path, encoding='utf-8') as f:
    nb_node = nbformat.read(f, as_version=4)

body, _ = HTMLExporter().from_notebook_node(nb_node)

with open(out_path, 'w', encoding='utf-8') as f:
    f.write(body)

print(f'Guardado: {out_path} ({len(body)//1024} KB)')